In [2]:
import pandas as pd
import requests

def fetch_fema_claims_by_county(fips_code):
    # Updated to v2 endpoint
    base_url = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
    
    params = {
        "$filter": f"countyCode eq '{fips_code}'",
        "$top": 1000
    }
    
    res = requests.get(base_url, params=params)
    res.raise_for_status()
    
    # In v2, response JSON root array is also 'FimaNfipClaims'
    data = res.json()
    return pd.DataFrame(data.get('FimaNfipClaims', []))

# Test again with your Van Buren County FIPS (19177)
sample_claims = fetch_fema_claims_by_county('19177')
print(f"Retrieved {len(sample_claims)} claims.")

Retrieved 53 claims.


In [6]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import os

def load_noaa_events(csv_path='noaa_21-25_with_huc_08.csv'):
    """Loads NOAA dataset and constructs proper 5-digit county FIPS codes."""
    df = pd.read_csv(csv_path)
    
    # Standardize 5-digit FIPS code
    df['FIPS_5'] = (
        df['STATE_FIPS'].astype(str).str.zfill(2) + 
        df['CZ_FIPS'].astype(str).str.zfill(3)
    )
    
    # Parse dates to datetime objects for temporal querying
    df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'])
    df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'])
    return df

def query_fema_claims_multi_fips(fips_list, start_date, end_date, date_buffer_days=14):
    """
    Queries OpenFEMA v2 NFIP claims across all counties impacted by the episode
    within the episode's overall date window.
    """
    base_url = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
    
    # Apply date window buffer
    search_start = (start_date - timedelta(days=2)).strftime('%Y-%m-%d')
    search_end = (end_date + timedelta(days=date_buffer_days)).strftime('%Y-%m-%d')
    
    # Format FIPS filter for multiple counties
    fips_conditions = " or ".join([f"countyCode eq '{fips}'" for fips in fips_list])
    
    fema_filter = (
        f"({fips_conditions}) and "
        f"dateOfLoss ge {search_start}T00:00:00.000Z and "
        f"dateOfLoss le {search_end}T23:59:59.000Z"
    )
    
    params = {
        "$filter": fema_filter,
        "$top": 10000
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        records = response.json().get('FimaNfipClaims', [])
        return pd.DataFrame(records)
    except Exception as e:
        print(f"Error querying OpenFEMA API: {e}")
        return pd.DataFrame()

def run_pipeline():
    df = load_noaa_events()
    
    print("==================================================")
    print(" NOAA Episode -> OpenFEMA Claims Search Pipeline")
    print("==================================================")
    
    episodes_summary = df.groupby('NEW_EPISODE_ID').agg(
        event_types=('EVENT_TYPE', lambda x: ', '.join(x.unique())),
        counties=('CZ_NAME', lambda x: ', '.join(x.unique())),
        start_date=('BEGIN_DT', 'min'),
        end_date=('END_DT', 'max'),
        event_count=('EVENT_ID', 'count')
    ).reset_index()
    
    print("\nSample Episodes from CSV:\n")
    print(episodes_summary[['NEW_EPISODE_ID', 'event_types', 'counties', 'event_count']].head(10).to_string(index=False))
    print("\n--------------------------------------------------")
    
    user_input = input("\nEnter the NEW_EPISODE_ID you want to inspect (or press Enter for default '191899_0'): ").strip()
    if not user_input:
        user_input = "191899_0"
        
    episode_rows = df[df['NEW_EPISODE_ID'] == user_input]
    if episode_rows.empty:
        print(f"NEW_EPISODE_ID '{user_input}' not found in the dataset.")
        return
        
    fips_list = episode_rows['FIPS_5'].unique().tolist()
    county_names = episode_rows['CZ_NAME'].unique().tolist()
    huc8_names = episode_rows['NAME'].unique().tolist()
    
    min_date = episode_rows['BEGIN_DT'].min()
    max_date = episode_rows['END_DT'].max()
    
    print("\n--------------------------------------------------")
    print(f"Selected Episode Summary:")
    print(f" • Episode ID:    {user_input}")
    print(f" • Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}")
    print(f" • Counties ({len(fips_list)}): {', '.join(county_names)} (FIPS: {', '.join(fips_list)})")
    print(f" • HUC-8 Basins:  {', '.join(huc8_names)}")
    print(f" • Date Window:   {min_date.strftime('%Y-%m-%d %H:%M')} to {max_date.strftime('%Y-%m-%d %H:%M')}")
    print("--------------------------------------------------\n")
    
    print("Searching OpenFEMA for matching claims across all impacted counties...")
    claims_df = query_fema_claims_multi_fips(
        fips_list=fips_list, 
        start_date=min_date, 
        end_date=max_date
    )
    
    if claims_df.empty:
        print("No FEMA NFIP claims found in OpenFEMA for these counties around this episode window.")
    else:
        print(f"Found {len(claims_df)} matching claim(s)!\n")
        
        cols_to_show = [col for col in ['dateOfLoss', 'countyCode', 'amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim'] if col in claims_df.columns]
        print(claims_df[cols_to_show].head(10).to_string(index=False))
        
        if 'amountPaidOnBuildingClaim' in claims_df.columns:
            tot_bldg = claims_df['amountPaidOnBuildingClaim'].sum()
            tot_cnt = claims_df['amountPaidOnContentsClaim'].sum() if 'amountPaidOnContentsClaim' in claims_df.columns else 0
            print("\nFinancial Totals for Matching Episode Claims:")
            print(f" • Building Payouts: ${tot_bldg:,.2f}")
            print(f" • Contents Payouts: ${tot_cnt:,.2f}")
            print(f" • Total Payouts:    ${(tot_bldg + tot_cnt):,.2f}")
            
        # CSV Export Prompt
        export_choice = input("\nWould you like to export these claims to a CSV file? (y/n): ").strip().lower()
        if export_choice in ['y', 'yes']:
            filename = f"fema_claims_episode_{user_input.replace('/', '_')}.csv"
            claims_df.to_csv(filename, index=False)
            print(f"✓ Successfully exported {len(claims_df)} claims to '{filename}'!")

if __name__ == "__main__":
    run_pipeline()

 NOAA Episode -> OpenFEMA Claims Search Pipeline

Sample Episodes from CSV:

NEW_EPISODE_ID event_types             counties  event_count
      157601_0       Flood            VAN BUREN            1
      158463_0 Flash Flood JEFFERSON, VAN BUREN            4
      158466_0 Flash Flood                 LINN            1
      159463_0 Flash Flood       LUCAS, WAPELLO            3
      159795_0 Flash Flood            ALLAMAKEE            1
      160825_0 Flash Flood                STORY            2
      161012_0 Flash Flood            VAN BUREN            1
      161088_0 Flash Flood                WORTH            4
      161208_0 Flash Flood           DES MOINES            2
      161640_0 Flash Flood                 IOWA            1

--------------------------------------------------

--------------------------------------------------
Selected Episode Summary:
 • Episode ID:    191899_0
 • Event Types:   Flood
 • Counties (11): LYON, SIOUX, OSCEOLA, DICKINSON, CHEROKEE, O'BRIEN, P